In [3]:
import boto3
import json
import datetime

# MODEL_ID = "INSERT INFERENCE PROFILE ARN HERE"
# GUARDRAIL_ID = "xnlg9ddiz850"
# MODEL_ID = "us.anthropic.claude-sonnet-4-6"
MODEL_ID = "us.anthropic.claude-sonnet-4-6"
# bedrock = boto3.client(service_name="bedrock-runtime", region_name="us-east-1")
from IPython.display import display, JSON

bedrock_runtime = boto3.client("bedrock-runtime", region_name="us-east-1")

bedrock_agent = boto3.client("bedrock-agent", region_name="us-east-1")
bedrock_agent_runtime = boto3.client("bedrock-agent-runtime", region_name="us-east-1")

In [9]:
prompt_name = f"job-description-{datetime.datetime.now().strftime('%Y%m%d%H%M%S')}"

template_text = """
You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words
"""
response = bedrock_agent.create_prompt(
    name=prompt_name,
    description="Generates inclusive job descriptions from structured inputs",
    defaultVariant="v1",
    variants=[
        {
            "name": "v1",
            "modelId": MODEL_ID,
            "templateType": "TEXT",
            "templateConfiguration": {
                "text": {
                    "inputVariables": [
                        {"name": "job_title"},
                        {"name": "responsibilities"},
                        {"name": "requirements"},
                        {"name": "location"},
                        {"name": "work_type"},
                    ],
                    "text": template_text,
                }
            },
            "inferenceConfiguration": {
                "text": {
                    "maxTokens": 500,
                    "temperature": 0.7,
                    # "topP": 0.9,
                    "stopSequences": [],
                }
            },
        }
    ],
)
print("\n==================== Response Object ====================\n")
display(JSON(response))

print("\n==================== Prompt ARN ====================\n")
print(response["arn"])


==================== Response Object ====================



<IPython.core.display.JSON object>


==================== Prompt ARN ====================

arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2


In [10]:
response = bedrock_agent.create_prompt_version(
    description="Initial prompt for creating job description documents.",
    promptIdentifier="5DOD4HGZI2",
)

print("\n==================== PROMPT VERSION ARN ====================\n")
print(response["arn"])


==================== PROMPT VERSION ARN ====================

arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2:1


In [11]:
prompt_arn_with_version = "arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2:1"

response = bedrock_runtime.converse(
    modelId=prompt_arn_with_version,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {
            "text": "Design user interfaces, run usability testing, collaborate with product teams"
        },
        "requirements": {
            "text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"
        },
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"},
    },
)

print("\n==================== Response Text ====================\n")
print(response["output"]["message"]["content"][0]["text"])


==================== Response Text ====================

# UX Designer — Full-Time | New York or Remote

## About the Role
We are looking for a talented and collaborative **UX Designer** to help shape intuitive, user-centered digital experiences. You will play a key role in our product development process, working closely with cross-functional teams to deliver thoughtful and impactful designs.

---

## What You'll Do
- Design and iterate on user interfaces that are accessible, functional, and visually engaging
- Plan and conduct usability testing to gather insights and inform design decisions
- Collaborate closely with product, engineering, and stakeholder teams throughout the design lifecycle
- Translate user needs and business goals into clear, effective design solutions

---

## What You'll Bring
- 3+ years of experience in UX or product design
- Proficiency in **Figma** for wireframing, prototyping, and design systems
- Working knowledge of **HTML/CSS** to communicate effectively 

In [16]:
def handle_response_stream(response):
    try:
        event_stream = response["optimizedPrompt"]
        for event in event_stream:
            if "optimizedPromptEvent" in event:
                print("\n==================== OPTIMIZED PROMPT ====================\n")
                print(
                    event["optimizedPromptEvent"]["optimizedPrompt"]["textPrompt"][
                        "text"
                    ]
                )
    except Exception as e:
        raise e


# OptimizePrompt requires a specific supported base model ID (not an inference profile
# like us.anthropic.claude-sonnet-4-6). See supported targetModelId values in the AWS docs.
# OPTIMIZE_TARGET_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"
# OPTIMIZE_TARGET_MODEL_ID = "anthropic.claude-3-5-sonnet-20241022-v2:0"
OPTIMIZE_TARGET_MODEL_ID = "anthropic.claude-3-haiku-20240307-v1:0"

prompt_input = {"textPrompt": {"text": template_text}}

response = bedrock_agent_runtime.optimize_prompt(
    input=prompt_input, targetModelId=OPTIMIZE_TARGET_MODEL_ID
)

print("\n==================== ORIGINAL PROMPT ====================\n")
print(template_text)
handle_response_stream(response)


==================== ORIGINAL PROMPT ====================


You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words


==================== OPTIMIZED PROMPT ====================

"<role>You are an expert HR assistant specializing in crafting compelling, professional, and inclusive job descriptions.</role>\n\n<task>Create a job description that attracts diverse, qualified candidates while clearly communicating role expectations and company culture.</task>\n\n<inputs>\n<job_title>{{job_title}}</job_title>\n<responsibilities>{{responsibilities}}</responsibilities>\n<requirements>{{requirements}}</requirements>\n<location>{{location}}</location>\n<work_type>{{work_type}}</work_type>\n</inputs>\n\n<instructions>\nCo

In [17]:
prompt_identifier = "5DOD4HGZI2"

existing_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print("\n==================== ORIGINAL PROMPT ====================\n")
print(existing_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


updated_variants = existing_prompt["variants"]
for variant in updated_variants:
    if variant["templateType"] == "TEXT":
        variant["templateConfiguration"]["text"][
            "text"
        ] = '<role>You are an expert HR assistant specializing in crafting compelling, professional, and inclusive job descriptions.</role>\n\n<task>Create a job description that attracts diverse, qualified candidates while clearly communicating role expectations and company culture.</task>\n\n<inputs>\n<job_title>{{job_title}}</job_title>\n<responsibilities>{{responsibilities}}</responsibilities>\n<requirements>{{requirements}}</requirements>\n<location>{{location}}</location>\n<work_type>{{work_type}}</work_type>\n</inputs>\n\n<instructions>\nCompose a professional job description following these guidelines:\n\n1. **Opening Summary** (2-3 sentences):\n   - Begin with an engaging overview of the role\n   - Highlight the position\'s impact and purpose\n   - Set a welcoming, professional tone\n\n2. **Key Responsibilities**:\n   - Present the main duties from <responsibilities> in a clear, organized manner\n   - Use action-oriented language (e.g., "lead," "develop," "collaborate")\n   - Prioritize the most important responsibilities first\n\n3. **Qualifications & Requirements**:\n   - List essential qualifications from <requirements>\n   - Distinguish between "required" and "preferred" qualifications when applicable\n   - Focus on skills and competencies rather than years of experience alone\n\n4. **Work Details**:\n   - Clearly state the <location> and <work_type> (remote, hybrid, on-site)\n   - Include any relevant logistical information\n\n5. **Inclusive Language Standards**:\n   - Avoid gendered pronouns; use "they/them" or "the candidate"\n   - Eliminate age-related terms (e.g., "digital native," "recent graduate")\n   - Remove unnecessary jargon or cultural idioms that may exclude candidates\n   - Use neutral terms like "team member" instead of "rockstar" or "ninja"\n   - Avoid phrases that may discourage underrepresented groups (e.g., "aggressive," "dominant")\n\n6. **Length & Style**:\n   - Keep the total description under 250 words\n   - Use clear, concise sentences\n   - Employ bullet points for easy scanning where appropriate\n   - Maintain a professional yet approachable tone\n</instructions>\n\n<output_format>\nProvide the complete job description immediately without any preamble or meta-commentary. Structure it with clear sections and formatting that is ready to post.\n</output_format>'


response = bedrock_agent.update_prompt(
    promptIdentifier=prompt_identifier,
    name=existing_prompt["name"],
    description=existing_prompt["description"],
    defaultVariant=existing_prompt["defaultVariant"],
    variants=updated_variants,
)

print("\n==================== UPDATED PROMPT ====================\n")

updated_prompt = bedrock_agent.get_prompt(promptIdentifier=prompt_identifier)
print(updated_prompt["variants"][0]["templateConfiguration"]["text"]["text"])


==================== ORIGINAL PROMPT ====================


You are an HR assistant.

Write a professional, inclusive job description using the following inputs:

Job title: {{job_title}}
Responsibilities: {{responsibilities}}
Requirements: {{requirements}}
Location: {{location}}
Work type: {{work_type}}

- Start with a clear summary
- Use concise, inclusive language
- Keep it under 250 words


==================== UPDATED PROMPT ====================

<role>You are an expert HR assistant specializing in crafting compelling, professional, and inclusive job descriptions.</role>

<task>Create a job description that attracts diverse, qualified candidates while clearly communicating role expectations and company culture.</task>

<inputs>
<job_title>{{job_title}}</job_title>
<responsibilities>{{responsibilities}}</responsibilities>
<requirements>{{requirements}}</requirements>
<location>{{location}}</location>
<work_type>{{work_type}}</work_type>
</inputs>

<instructions>
Compose a professi

In [18]:
response = bedrock_agent.create_prompt_version(
    description="Optimized prompt for creating job description documents.",
    promptIdentifier=prompt_identifier,
)

print("\n==================== PROMPT VERSION ARN ====================\n")
print(response["arn"])


==================== PROMPT VERSION ARN ====================

arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2:2


In [19]:
prompt_arn = "arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2:1"

response = bedrock_runtime.converse(
    modelId=prompt_arn,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {
            "text": "Design user interfaces, run usability testing, collaborate with product teams"
        },
        "requirements": {
            "text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"
        },
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"},
    },
)

print("\\n==================== Response Text ====================\\n")
print(response["output"]["message"]["content"][0]["text"])

\n==================== Response Text ====================\n
# UX Designer — Full-Time | New York or Remote

## About the Role
We are looking for a talented and collaborative **UX Designer** to help shape intuitive, user-centered digital experiences. You will work closely with cross-functional teams to design products that are both functional and delightful to use.

---

## What You'll Do
- Design clear, accessible, and visually engaging user interfaces
- Plan and conduct usability testing to gather meaningful user insights
- Collaborate with product, engineering, and stakeholder teams throughout the design process
- Iterate on designs based on feedback and data

---

## What You'll Bring
- **3+ years** of professional UX or UI design experience
- Proficiency in **Figma** for wireframing and prototyping
- Working knowledge of **HTML/CSS** to communicate effectively with developers
- Strong verbal and written **communication skills**
- A collaborative mindset and openness to feedback

--

In [20]:
prompt_arn = "arn:aws:bedrock:us-east-1:623271127785:prompt/5DOD4HGZI2:2"

response = bedrock_runtime.converse(
    modelId=prompt_arn,
    promptVariables={
        "job_title": {"text": "UX Designer"},
        "responsibilities": {
            "text": "Design user interfaces, run usability testing, collaborate with product teams"
        },
        "requirements": {
            "text": "3+ years experience, Figma, HTML/CSS knowledge, communication skills"
        },
        "location": {"text": "New York or remote"},
        "work_type": {"text": "Full-time"},
    },
)

print("\\n==================== Response Text ====================\\n")
print(response["output"]["message"]["content"][0]["text"])

\n==================== Response Text ====================\n
# UX Designer

**Location:** New York, NY (or Remote) | **Type:** Full-Time

---

We're looking for a thoughtful and skilled UX Designer to shape meaningful digital experiences for our users. In this role, you'll sit at the intersection of design, research, and product strategy — directly influencing how people interact with our platform every day.

---

## Key Responsibilities

- Design intuitive, accessible user interfaces across web and mobile products
- Lead usability testing sessions and translate findings into actionable design improvements
- Collaborate closely with product managers, engineers, and stakeholders throughout the development lifecycle
- Maintain and contribute to a consistent design system and component library
- Present design concepts clearly and incorporate feedback iteratively

---

## Qualifications

**Required:**
- Demonstrated experience designing user-centered digital products (portfolio required)
-

## Clean up by Deletion

In [21]:
response = bedrock_agent.list_prompts()

for prompt in response["promptSummaries"]:
    prompt_id = prompt["id"]  # The correct key is 'id', not 'promptId'
    print(f"Deleting prompt: {prompt['name']} (ID: {prompt_id})")
    bedrock_agent.delete_prompt(promptIdentifier=prompt_id)

print("All prompts deleted successfully")

Deleting prompt: job-description-20260813093337 (ID: 1243MI7FGH)
Deleting prompt: job-description-20260813100908 (ID: 6JWJCW4HVU)
Deleting prompt: job-description-20260813101020 (ID: ZI5GAEEMF9)
Deleting prompt: job-description-20260813101104 (ID: FZTJE9EOZ0)
Deleting prompt: job-description-20260813102741 (ID: 5DOD4HGZI2)
All prompts deleted successfully
